<a href="https://colab.research.google.com/github/S-Y-3-D/generative-forge/blob/Autoregressive-models/notebooks/autoregressive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autoregressive Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/S-Y-3-D/generative-forge/blob/main/notebooks/autoregressive.ipynb)

**Experiment:**  
**Hypothesis:**  
**Status:** 🟡 In Progress

### Dataset

In [1]:
!pip install kagglehub

In [2]:
import kagglehub
from google.colab import userdata
import json
import re
import string
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from collections import Counter

In [3]:
print(userdata.get('KAGGLE_HUB_API_KEY'))

KGAT_d20b2bbf811945efd1f99c720331438e


In [4]:
kagglehub.login()

Kaggle credentials set.


In [8]:
kagglehub.dataset_download("hugodarwood/epirecipes")

Using Colab cache for faster access to the 'epirecipes' dataset.


'/kaggle/input/epirecipes'

In [9]:
with open("/kaggle/input/epirecipes/full_format_recipes.json") as json_data:
  recipe_data = json.load(json_data)

filtered_data = [
'Recipe for ' + x['title']+ ' | ' + ' '.join(x['directions'])
for x in recipe_data
if 'title' in x
and x['title'] is not None
and 'directions' in x
and x['directions'] is not None
]

In [10]:
filtered_data[0]

'Recipe for Lentil, Apple, and Turkey Wrap  | 1. Place the stock, lentils, celery, carrot, thyme, and salt in a medium saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool. 2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper. 3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the lettuce. Roll up the lavash, slice crosswise, and serve. If using tortillas, spread the lentils in the center, top with the turkey and lettuce, and fold up the bottom, left side, and right side before rolling away from you.'

### DataLoader

In [11]:
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r' \1 ', s)
    s = re.sub(' +', ' ', s)
    return s

In [12]:
def tokenize(text):
  return text.lower().split()

In [13]:
text_data = [tokenize(pad_punctuation(x)) for x in filtered_data]

In [14]:
counter = Counter()
for text in text_data:
  counter.update(text)

In [15]:
max_vocab_size = 10000
vocab = ["","[UNK]"] + [word for word, _ in counter.most_common(max_vocab_size - 2)]
word_to_index = {word: idx for idx, word in enumerate(vocab)}

In [16]:
word_to_index

{'': 0,
 '[UNK]': 1,
 '.': 2,
 ',': 3,
 'and': 4,
 'to': 5,
 'in': 6,
 'the': 7,
 'with': 8,
 'a': 9,
 'until': 10,
 '1': 11,
 'minutes': 12,
 '-': 13,
 'of': 14,
 '2': 15,
 'for': 16,
 'heat': 17,
 'add': 18,
 'about': 19,
 'over': 20,
 'bowl': 21,
 ';': 22,
 '/': 23,
 'salt': 24,
 'into': 25,
 'recipe': 26,
 '|': 27,
 'on': 28,
 'medium': 29,
 'large': 30,
 'mixture': 31,
 '4': 32,
 'pepper': 33,
 '(': 34,
 ')': 35,
 '3': 36,
 'oil': 37,
 'is': 38,
 'water': 39,
 'transfer': 40,
 'or': 41,
 'stir': 42,
 'cook': 43,
 'pan': 44,
 'remaining': 45,
 'then': 46,
 'oven': 47,
 'stirring': 48,
 'cover': 49,
 'butter': 50,
 'from': 51,
 'cup': 52,
 'inch': 53,
 'sauce': 54,
 'skillet': 55,
 'sugar': 56,
 'at': 57,
 'baking': 58,
 '5': 59,
 'cool': 60,
 'it': 61,
 'be': 62,
 'season': 63,
 'place': 64,
 'small': 65,
 'each': 66,
 'let': 67,
 'serve': 68,
 'boil': 69,
 'simmer': 70,
 'remove': 71,
 'top': 72,
 'whisk': 73,
 'cut': 74,
 'high': 75,
 'cream': 76,
 'ahead': 77,
 'heavy': 78,
 'ar

In [17]:
output_sequence_length = 201
def vectorize(text, seq_len=output_sequence_length):
    tokens = tokenize(text)[:seq_len]
    indices = [word_to_index.get(t, 1) for t in tokens]
    indices += [0] * (seq_len - len(indices))
    return torch.tensor(indices, dtype=torch.long)

In [18]:
index_to_word = {idx: word for idx, word in enumerate(vocab)}

In [19]:
def detokenize(indices):
  indices = indices.numpy().tolist()
  return ' '.join([index_to_word[i] for i in indices])

In [20]:
class TextDataset(Dataset):
  def __init__(self, texts):
    self.data = [vectorize(t) for t in texts]


  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    sequence = self.data[idx]
    x = sequence[:-1]   # input:  tokens 0→199
    y = sequence[1:]    # target: tokens 1→200 (shifted by 1)
    return x, y

In [21]:
dataset = TextDataset(filtered_data)

In [22]:
filtered_data[0], dataset[0]

('Recipe for Lentil, Apple, and Turkey Wrap  | 1. Place the stock, lentils, celery, carrot, thyme, and salt in a medium saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool. 2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper. 3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the lettuce. Roll up the lavash, slice crosswise, and serve. If using tortillas, spread the lentils in the center, top with the turkey and lettuce, and fold up the bottom, left side, and right side before rolling away from you.',
 (tensor([  26,   16,    1,    1,    4,  221,  212,   27,    1,   64,    7,    1,
             1,    1,

In [23]:
train_size = int(0.9 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32)

### LSTM Training

In [27]:
from torch import nn
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

class Autoregressive(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.LSTM = nn.LSTM(100, 100)
    self.linear = nn.Linear(100, vocab_size)

  def forward(self, x):
    x = self.embedding(x)
    x, _ = self.LSTM(x)
    x = self.linear(x)
    return x

In [30]:
model = Autoregressive(10000)
criterion = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)
epochs = 10

In [ ]:
for epoch in range(epochs):
    total_loss = 0
    for batch in train_dataloader:
        optimizer.zero_grad()
        x, y = batch
        response = model(x)
        loss = criterion(response.view(-1, 10000), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} loss: {total_loss / len(train_dataloader):.4f}")